# 📊 Projeto G1 — E-commerce Brasil
## Análise e Visualização de Dados com Python

**Problema:** compreender o desempenho de um e-commerce brasileiro entre 2015 e 2024, identificando padrões de faturamento, lucro, canais, categorias, produtos, regiões e experiência do cliente.

**Objetivos**
- Preparar e validar a base;
- Criar atributos derivados;
- Construir KPIs;
- Explorar padrões temporais e geográficos;
- Comparar categorias, produtos e canais;
- Investigar correlações;
- Produzir conclusões para apoio à decisão.


## 2. Base de dados

A base simulada contém registros de vendas com informações temporais, geográficas, comerciais e operacionais.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("../dados/simulacao_ecommerce_brasil.csv")
df["data"] = pd.to_datetime(df["data"], errors="coerce")
df.head()


In [ ]:
print("Linhas:", len(df))
print("Colunas:", len(df.columns))
print("Período:", df["data"].min().date(), "a", df["data"].max().date())
print("\nDimensão:", df.shape)
print("\nTipos:")
print(df.dtypes)


## 3. Limpeza e preparação

In [ ]:
# Valores ausentes
print("Valores ausentes por coluna:")
display(df.isna().sum())

# Duplicidades
print("Registros duplicados:", df.duplicated().sum())

# Padronização de textos
for c in ["regiao","uf","cidade","canal_venda","categoria","produto"]:
    df[c] = df[c].astype("string").str.strip()

# Datas inválidas
print("Datas inválidas:", df["data"].isna().sum())


### Resultado
A base possui 4.440 registros e 16 variáveis. A análise inicial não encontrou valores ausentes nem duplicidades relevantes. A data foi convertida para `datetime` e os campos textuais foram padronizados.

## 4. Engenharia de atributos

In [ ]:
df["ano"] = df["data"].dt.year
df["mes"] = df["data"].dt.month
df["trimestre"] = df["data"].dt.quarter
df["ano_mes"] = df["data"].dt.to_period("M").astype(str)
df["margem_lucro"] = np.where(df.faturamento != 0, df.lucro/df.faturamento*100, 0)
df["ticket_medio"] = np.where(df.quantidade != 0, df.faturamento/df.quantidade, 0)

display(df[["data","ano","mes","trimestre","margem_lucro","ticket_medio"]].head())


## 5. Análise exploratória

In [ ]:
print("Regiões:", df.regiao.nunique())
print("UFs:", df.uf.nunique())
print("Cidades:", df.cidade.nunique())
print("Canais:", df.canal_venda.nunique())
print("Categorias:", df.categoria.nunique())
print("Produtos:", df.produto.nunique())

display(df.groupby("regiao").faturamento.sum().sort_values(ascending=False).to_frame("faturamento"))


## 6. KPIs

In [ ]:
kpis = pd.Series({
    "Faturamento total": df.faturamento.sum(),
    "Lucro total": df.lucro.sum(),
    "Margem": df.lucro.sum()/df.faturamento.sum()*100,
    "Unidades vendidas": df.quantidade.sum(),
    "Ticket médio por unidade": df.faturamento.sum()/df.quantidade.sum(),
    "Avaliação média": df.avaliacao_cliente.mean(),
    "Prazo médio": df.prazo_entrega.mean()
})
kpis


## 7. Evolução temporal

In [ ]:
mensal=df.groupby("data",as_index=False).agg(faturamento=("faturamento","sum"),lucro=("lucro","sum"))
plt.figure(figsize=(13,5))
plt.plot(mensal.data,mensal.faturamento,label="Faturamento")
plt.plot(mensal.data,mensal.lucro,label="Lucro")
plt.title("Evolução mensal")
plt.xlabel("Data"); plt.ylabel("R$"); plt.legend()
plt.xticks(rotation=30); plt.tight_layout(); plt.show()


In [ ]:
anual=df.groupby("ano").agg(faturamento=("faturamento","sum"),lucro=("lucro","sum"),quantidade=("quantidade","sum"))
anual["margem"]=anual.lucro/anual.faturamento*100
display(anual.style.format({"faturamento":"R$ {:,.2f}","lucro":"R$ {:,.2f}","quantidade":"{:,.0f}","margem":"{:.2f}%"}))


## 8. Comparações de negócio

In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
sns.barplot(data=df.groupby("regiao",as_index=False).faturamento.sum().sort_values("faturamento",ascending=False),
            x="faturamento",y="regiao",ax=ax)
ax.set_title("Faturamento por região"); ax.set_xlabel("R$"); ax.set_ylabel("")
plt.tight_layout(); plt.show()


In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
sns.barplot(data=df.groupby("categoria",as_index=False).faturamento.sum().sort_values("faturamento",ascending=False),
            x="faturamento",y="categoria",ax=ax)
ax.set_title("Faturamento por categoria"); ax.set_xlabel("R$"); ax.set_ylabel("")
plt.tight_layout(); plt.show()


In [ ]:
fig,ax=plt.subplots(figsize=(9,5))
sns.barplot(data=df.groupby("canal_venda",as_index=False).faturamento.sum().sort_values("faturamento",ascending=False),
            x="faturamento",y="canal_venda",ax=ax)
ax.set_title("Faturamento por canal"); ax.set_xlabel("R$"); ax.set_ylabel("")
plt.tight_layout(); plt.show()


## 9. Análise estatística

In [ ]:
numericas=["quantidade","preco_unitario","faturamento","custo","lucro","prazo_entrega","avaliacao_cliente"]
corr=df[numericas].corr()
plt.figure(figsize=(10,7))
sns.heatmap(corr,annot=True,fmt=".2f",cmap="coolwarm")
plt.title("Matriz de correlação")
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(10,5))
sns.scatterplot(data=df.sample(min(1200,len(df)),random_state=42),
                x="prazo_entrega",y="avaliacao_cliente",hue="canal_venda",alpha=.65)
plt.title("Prazo de entrega × avaliação")
plt.tight_layout(); plt.show()


## 10. Interpretação dos resultados

In [ ]:
top_reg=df.groupby("regiao").faturamento.sum().idxmax()
top_cat=df.groupby("categoria").faturamento.sum().idxmax()
top_prod=df.groupby("produto").faturamento.sum().idxmax()
top_canal=df.groupby("canal_venda").faturamento.sum().idxmax()
top_uf=df.groupby("uf").faturamento.sum().idxmax()

print(f"Região líder: {top_reg}")
print(f"Categoria líder: {top_cat}")
print(f"Produto líder: {top_prod}")
print(f"Canal líder: {top_canal}")
print(f"UF líder: {top_uf}")


### Interpretação

A análise permite observar a concentração de resultados por região, categoria, canal e produto. O ranking deve ser interpretado em conjunto com o lucro e a margem, pois faturamento elevado não significa necessariamente maior rentabilidade.

A análise temporal ajuda a identificar crescimento ou redução da atividade comercial. A matriz de correlação permite observar associações lineares entre variáveis numéricas, enquanto a relação entre prazo e avaliação fornece uma visão sobre a experiência do cliente.

Os resultados numéricos são calculados pelas células para evitar valores digitados manualmente e permitir a reprodução da análise.


## 11. Conclusão

O projeto atende ao fluxo de análise de dados solicitado: exploração, tratamento, engenharia de atributos, KPIs, visualizações, interpretação e conclusão. O dashboard Streamlit amplia a análise com filtros múltiplos, KPIs dinâmicos, tabelas e seções. A camada SQLite/SQLAlchemy adiciona persistência e modelagem relacional. Como evolução futura, podem ser incluídas previsão de vendas, dados reais de clientes e mapas geográficos.